# 02 — Preprocessing
**Anaemia Prediction using ML & Explainable AI**

> ⚠️ Educational decision-support prototype — **not** a diagnostic tool.

Turn the clean data into a **stratified train/test split** + a **saved scaler**.
This is the hand-off the model-training step consumes. All logic lives in `src/preprocess.py`; this notebook explains and verifies it.

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd, numpy as np, joblib
from src import config
from src.data_loader import load_clean
from src import preprocess as pp
print('Setup OK')

Setup OK


## 1. Load clean data
534 unique patients (887 duplicates already removed).

In [2]:
df = load_clean()
print('clean shape:', df.shape)
df.head()

clean shape: (534, 6)


,Gender,Hemoglobin,MCH,MCHC,MCV,Result
0,1,14.9,22.7,29.1,83.7,0
1,0,15.9,25.4,28.3,72.0,0
2,0,9.0,21.5,29.6,71.2,1
3,0,14.9,16.0,31.4,87.5,0
4,1,14.7,22.0,28.2,99.5,0


## 2. Stratified train/test split
- **Train set** (~80%): the model learns from this.
- **Test set** (~20%): held back, never seen in training — our honest exam.
- **Stratified**: keep the same anaemic/not-anaemic ratio in both halves.
- **Fixed random_state**: same split every run, so results are reproducible.

In [3]:
X_train, X_test, y_train, y_test = pp.split_data(df)
print('X_train:', X_train.shape, ' X_test:', X_test.shape)
print('\nClass balance preserved?')
print('  full :', (df[config.TARGET].value_counts(normalize=True)*100).round(1).to_dict())
print('  train:', (y_train.value_counts(normalize=True)*100).round(1).to_dict())
print('  test :', (y_test.value_counts(normalize=True)*100).round(1).to_dict())

X_train: (427, 5)  X_test: (107, 5)

Class balance preserved?
  full : {0: 53.7, 1: 46.3}
  train: {0: 53.6, 1: 46.4}
  test : {0: 54.2, 1: 45.8}


## 3. Scaling (fit on TRAIN only)
`StandardScaler` rescales each feature to **mean 0, std 1**. SVM needs this because it compares distances; a big-range feature would otherwise dominate. We fit the scaler on **training data only** so no test information leaks in.

In [4]:
scaler = pp.fit_scaler(X_train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=config.FEATURES)
before = X_train[config.FEATURES].agg(['mean','std']).round(2).T
before.columns = ['mean_before','std_before']
after  = X_train_scaled.agg(['mean','std']).round(2).T
after.columns = ['mean_after','std_after']
comp = pd.concat([before, after], axis=1)
print('Feature stats BEFORE vs AFTER scaling:')
comp

Feature stats BEFORE vs AFTER scaling:


,mean_before,std_before,mean_after,std_after
Gender,0.51,0.50,0.0,1.0
Hemoglobin,13.30,2.10,0.0,1.0
MCH,22.97,3.90,0.0,1.0
MCHC,30.28,1.42,-0.0,1.0
MCV,85.57,9.59,-0.0,1.0


## 4. Persist the hand-off artifacts
Save splits (original units) + `scaler.pkl`. The training step loads these.

In [5]:
pp.build_and_save()
for p in [config.X_TRAIN_PATH, config.X_TEST_PATH, config.Y_TRAIN_PATH, config.Y_TEST_PATH, config.SCALER_PATH]:
    print(('OK ' if os.path.exists(p) else 'MISSING '), p.name)

OK  X_train.csv
OK  X_test.csv
OK  y_train.csv
OK  y_test.csv
OK  scaler.pkl


## 5. Confirm the model-ready loader
`load_processed(scaled=True)` is the one call the trainer uses — scaled features, mean~0/std~1.

In [6]:
Xtr, Xte, ytr, yte = pp.load_processed(scaled=True)
print('scaled train means (~0):', Xtr.mean().round(2).to_dict())
print('scaled train stds  (~1):', Xtr.std().round(2).to_dict())
print('shapes ->', Xtr.shape, Xte.shape)

scaled train means (~0): {'Gender': 0.0, 'Hemoglobin': 0.0, 'MCH': 0.0, 'MCHC': -0.0, 'MCV': -0.0}
scaled train stds  (~1): {'Gender': 1.0, 'Hemoglobin': 1.0, 'MCH': 1.0, 'MCHC': 1.0, 'MCV': 1.0}
shapes -> (427, 5) (107, 5)


## 6. Summary / hand-off contract
- **534 patients -> 427 train / 107 test**, stratified (class ratio preserved).
- Scaler fit on **train only**, saved to `models/scaler.pkl`.
- Artifacts in `data/processed/` are the inputs the **model-training step** consumes.
- Trainer just calls `preprocess.load_processed(scaled=True)` -> fits RF / SVM / XGBoost
  on the **identical split** and saves the three `.pkl` models.
- Next: **Phase 4 — training** (handled by teammate via `src/train.py`).